# PCA by SVD

Principal component analysis finds the directions along which data varies
most. Center the data matrix X, with one sample per row, and take its singular
value decomposition Xc = U Σ Vᵀ. The columns of V are the directions, the
principal components. The squared singular values measure the variance along
each. Keeping the first k components gives the closest rank-k approximation of
the data. The tutorial's data set is small enough to see all of it: 8 samples
of 3 features, the third built to follow the first.

**Uses:** `mtl5.svd`, `mtl5.eigvalsh`, `mtl5.condition_number`.

**From C++:** [`phase06_eigenvalue_svd/pca_svd.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase06_eigenvalue_svd/pca_svd.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## The data, centered

PCA measures variation about the mean, so each feature's mean comes off first.

In [ ]:
X = np.array(
    [
        [1.0, 4.2, 1.1],
        [2.1, 3.8, 2.3],
        [3.0, 4.5, 2.8],
        [4.2, 3.1, 4.4],
        [5.1, 4.8, 5.0],
        [5.8, 3.3, 6.1],
        [7.0, 4.1, 6.8],
        [8.1, 3.6, 8.3],
    ]
)
m, p = X.shape
means = X.mean(axis=0)
Xc = X - means
print("column means:", np.round(means, 3))
assert np.allclose(Xc.sum(axis=0), 0.0, atol=1e-14)

## The decomposition

`mtl5.svd` returns U, the singular values s as a vector in descending order,
and V, whose columns are the principal components. It is the full SVD: U is
m × m, and only its first p columns pair with a singular value. The fraction of the
variance along component k is sₖ² / Σ sⱼ². The first component carries almost
all of it, because features 1 and 3 move together.

In [ ]:
U, s, V = mtl5.svd(mtl5.matrix(Xc))
explained = s**2 / np.sum(s**2)
for kk, (sigma, share) in enumerate(zip(s, explained), start=1):
    print(f"sigma_{kk} = {sigma:.4f}   variance explained {100 * share:5.1f}%")
print("principal components (columns):\n", np.round(V, 4))

assert U.shape == (m, m)
assert np.allclose(U[:, :p] @ np.diag(s) @ V.T, Xc, atol=1e-13)  # Xc = U S V^T
assert np.allclose(V.T @ V, np.eye(p), atol=1e-13)  # orthonormal directions
assert (np.diff(s) <= 0).all()  # descending
assert explained[0] > 0.9

The first component is close to (1, 0, 1) / √2, up to sign: the direction in
which features 1 and 3 grow together. The second is close to feature 2 alone.

## The covariance gives the same variances

The covariance matrix is C = XcᵀXc / (m − 1). Its eigenvalues are the
variances along the principal components, sₖ² / (m − 1). The two routes agree
here, but the SVD is the one to use: forming XcᵀXc squares the condition
number, the same loss as the normal equations in the
[curve fit](factorizations.ipynb).

In [ ]:
C = Xc.T @ Xc / (m - 1)
cov_eigs = mtl5.eigvalsh(mtl5.matrix(C))  # ascending
svd_vars = np.sort(s**2 / (m - 1))
for a, b in zip(svd_vars, cov_eigs):
    print(f"s^2/(m-1) = {a:.6f}   eigenvalue of C = {b:.6f}   difference {abs(a - b):.1e}")
assert np.allclose(svd_vars, cov_eigs, rtol=1e-12, atol=0)

k_xc, k_c = mtl5.condition_number(mtl5.matrix(Xc)), mtl5.condition_number(mtl5.matrix(C))
print(f"cond(Xc) = {k_xc:.1f}, cond(C) = {k_c:.1f} = cond(Xc)^2")
assert np.isclose(k_c, k_xc**2, rtol=1e-10)

## From three dimensions to two

Projecting onto the first two components, Xc V₂, gives each sample two
coordinates instead of three. Mapping them back, Xc V₂ V₂ᵀ plus the means,
gives the closest data to X that lies in a plane. How close is exact: by the
Eckart–Young theorem, the error in the Frobenius norm is the dropped singular
value, s₃.

In [ ]:
V2 = V[:, :2]
scores = Xc @ V2  # m x 2: the data in the plane of the first two components
X_approx = scores @ V2.T + means

error = np.linalg.norm(X - X_approx, "fro")
relative = error / np.linalg.norm(Xc, "fro")
print(f"||X - X_approx||_F = {error:.6f}   s_3 = {s[2]:.6f}")
print(f"relative error {100 * relative:.2f}%, variance kept {100 * explained[:2].sum():.1f}%")

assert np.isclose(error, s[2], rtol=1e-12)
assert np.isclose(relative**2, explained[2], rtol=1e-12)  # the variance not kept

Two views of the result. On the left, the samples in features 1 and 3 with the
first component through their mean, and each sample's rank-1 approximation on
it. On the right, the variance each component explains.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8.4, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})

X1 = np.outer(Xc @ V[:, 0], V[:, 0]) + means  # rank-1 approximation
ax1.plot(X[:, 0], X[:, 2], "o", color=SERIES[0], label="samples")
ax1.plot(X1[:, 0], X1[:, 2], "x", color=SERIES[1], label="rank-1 approximation")
for a, b in zip(X, X1):
    ax1.plot([a[0], b[0]], [a[2], b[2]], color=INK_2, linewidth=0.6)
t = np.linspace(-5.0, 5.0, 2)
line = means + np.outer(t, V[:, 0])
ax1.plot(line[:, 0], line[:, 2], color=SERIES[1], linewidth=1, label="first component")
ax1.set_xlabel("feature 1")
ax1.set_ylabel("feature 3")
ax1.set_aspect("equal")
ax1.legend(frameon=False, fontsize=8)
ax1.grid(True)
ax1.set_title("Samples and the first component", fontsize=10)

ax2.bar([f"PC{kk}" for kk in range(1, p + 1)], 100 * explained, color=SERIES[0])
for x_pos, share in enumerate(explained):
    ax2.text(x_pos, 100 * share + 1.5, f"{100 * share:.1f}%", ha="center", fontsize=9)
ax2.set_ylabel("variance explained (%)")
ax2.set_ylim(0, 110)
ax2.set_title("Variance by component", fontsize=10)
plt.tight_layout()
plt.show()

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `svd(Xc, U, S, V)`, S a diagonal matrix | `U, s, V = mtl5.svd(Xc)`, s a vector, descending |
| column means in a loop | `X.mean(axis=0)` on the NumPy array |
| `trans(Xc) * Xc` | `Xc.T @ Xc` |
| `eigenvalue_symmetric(C)` | `mtl5.eigvalsh(C)`, ascending |
| `Xc * V2`, `X_reduced * trans(V2)` | `Xc @ V2`, `scores @ V2.T` |
| `frobenius_norm(Xc)` | `np.linalg.norm(Xc, "fro")`; `mtl5.frobenius_norm` takes only complex matrices so far |

mtl5's `V` has the principal components as columns, as MTL5's does. NumPy's
`np.linalg.svd` returns Vᵀ instead, with the components as rows.